# Text-to-image test: free open-source models on a free GPU

Checks whether open-licence (Apache-2.0) text-to-image models can make reel stills on **Colab free (T4)** or **Kaggle (T4 x2)**.
This is a standalone experiment. It doesn't touch the studio pipeline, and it doesn't use Google Drive.

**Colab (browser or the VS Code Colab extension):** pick a **T4 GPU** runtime, then *Run all*. Add an `HF_TOKEN` secret if you want FLUX.1 schnell: it is gated on Hugging Face, so accept its licence first. Z-Image and FLUX.2 klein don't need a token.
**Kaggle:** verify your phone, then set Accelerator **GPU T4 x2**, Internet **On**, and *Run all*.

**Colab vs Kaggle:**
- Free Colab has only **~12.7 GB system RAM**. Full-precision models (16–33 GB) would crash the kernel while loading.
- So on low-RAM machines the notebook loads every model **4-bit (NF4, bitsandbytes)** and skips FLUX.1 schnell (12B, too big even then).
- Kaggle (~30 GB RAM) tries full precision first.
- The smoke test saves after every model. If the kernel dies, re-run it and it skips the models that are already done.

The output is `t2i-test-results.zip`, holding the smoke-test images, 8 reel stills at 1080×1920 and `manifest.json`.
- In the browser, it downloads automatically.
- In VS Code, use **Colab: Download** on `/content/t2i-test-results.zip`.

Unzip it into `experiments/t2i-test/images/`.

| Step | What it does | Time on a T4 (expected) |
|---|---|---|
| 1 | Install + GPU report | ~3 min |
| 2 | Smoke test: 1 image per model, auto-falls back on dtype/offload | ~20–30 min (downloads dominate) |
| 3 | 8 reel stills with the winning model | ~10 min |
| 4 | Upscale to 1080×1920 + zip | ~2 min |

In [ ]:
# 1. Install (newest diffusers from source: Z-Image and FLUX.2 klein pipelines need it)
!pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece spandrel bitsandbytes psutil
# Kaggle ships an old torchao that breaks the newest diffusers' import (FqnToConfig). We don't use torchao.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Fail fast with the full error if any pipeline can't be imported (instead of 24 truncated FAIL lines later)
import importlib
for m in ["diffusers.pipelines.z_image.pipeline_z_image", "diffusers.pipelines.flux2.pipeline_flux2_klein",
          "diffusers.pipelines.flux.pipeline_flux"]:
    importlib.import_module(m); print("import ok:", m)

In [ ]:
import os, gc, json, time, platform, zipfile, traceback
import numpy as np, torch, diffusers, transformers, psutil
from PIL import Image

ON_KAGGLE = os.path.exists("/kaggle")
OUT = "/kaggle/working/t2i" if ON_KAGGLE else "/content/t2i"
os.makedirs(f"{OUT}/smoke", exist_ok=True); os.makedirs(f"{OUT}/reel", exist_ok=True)

# Optional HF token (only needed if a model becomes gated or you hit rate limits)
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: continuing anonymously")

N_GPU = torch.cuda.device_count()
RAM_GB = round(psutil.virtual_memory().total / 2**30, 1)
LOW_RAM = RAM_GB < 24          # free Colab (~12.7 GB): load 4-bit only, or the kernel dies while loading
ENV = {
    "ram_gb": RAM_GB, "low_ram_mode": LOW_RAM,
    "platform": "kaggle" if ON_KAGGLE else "colab",
    "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)],
    "vram_gb": [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1) for i in range(N_GPU)],
    "bf16_native": torch.cuda.is_bf16_supported() if N_GPU else False,
    "torch": torch.__version__, "diffusers": diffusers.__version__, "transformers": transformers.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=1))
assert N_GPU, "No GPU: switch the accelerator to a T4 and re-run"

## Config: models, house style, shots
Every model here is **Apache-2.0** (free for channel use). Non-commercial models (FLUX.1/FLUX.2 *dev*, SDXL-Turbo) are deliberately left out.
The style prefix follows the channel's `reference-flat` look: flat colour blocks in the brand palette, clean tech geometry, never cartoon, pastel or kids-style.

In [ ]:
MODELS = {
    # quant = the big components to load 4-bit on low-RAM machines
    "z-image-turbo": dict(repo="Tongyi-MAI/Z-Image-Turbo",           cls="ZImagePipeline",     steps=9, guidance=0.0,
                          quant=["transformer", "text_encoder"], licence="Apache-2.0"),
    "flux2-klein-4b": dict(repo="black-forest-labs/FLUX.2-klein-4B", cls="Flux2KleinPipeline", steps=4, guidance=1.0,
                          quant=["transformer", "text_encoder"], licence="Apache-2.0"),
    "flux1-schnell":  dict(repo="black-forest-labs/FLUX.1-schnell",  cls="FluxPipeline",       steps=4, guidance=0.0,
                          quant=["transformer", "text_encoder_2"], licence="Apache-2.0"),
}
SKIP = {"flux1-schnell"} if LOW_RAM else set()   # 12B + T5: its weight shards alone (~10 GB) can exhaust free Colab RAM
W, H = 720, 1280            # exact 9:16, multiples of 16; upscaled to 1080x1920 at the end
SEED = 1234
PASS_SECONDS, PASS_VRAM_GB = 90, 15

STYLE = ("Flat vector illustration, solid colour blocks, electric blue #2B3BE0, bright yellow #FFD23F, "
         "off-white #EEF1F6 and deep navy #0B1020 only, clean futuristic tech geometry, crisp edges, "
         "minimal, modern, high contrast, vertical 9:16 composition, no gradients, no grain, no glow, "
         "no text, no letters, no logos, no people, not cartoon, not pastel, not childish. ")

SMOKE_PROMPT = "A sleek smartphone floating over a perspective grid floor, a chat bubble rising from the screen."

# 8 shots adapted from productions/2026-10-09-tokenization/storyboard-reel.md (visual-only versions: no on-screen text)
SHOTS = [
    ("s01", "A sleek smartphone floating over a perspective grid floor, a chat bubble rising from the screen, a question-mark shape glowing."),
    ("s02", "A giant blinking text cursor beside a huge abstract glyph block that is turning into a stack of binary bars, camera close-up."),
    ("s03", "A tidy 2 by 5 grid of square tiles, each tile flipping from a letter-like shape into a numeric bar pattern, three tiles ringed in yellow."),
    ("s04", "A long horizontal ribbon of small tiles scrolling into the distance, endless and overwhelming, deep perspective."),
    ("s05", "Tall server-rack shelves filled with thousands of small coloured tiles, like a giant library of pieces, receding into the distance."),
    ("s06", "A glowing horizontal scan beam slicing a glass bar into seven pieces with small sparks at each cut."),
    ("s07", "A tall glass panel listing rows of data, one single row highlighted in yellow, a pointer arrow aimed at it."),
    ("s08", "A glass meter filling with small blue tiles dropping in from above, a gauge needle climbing, futuristic dashboard."),
]
print(len(SHOTS), "shots;", "models:", list(MODELS))

## Helpers: every attempt runs in its own child process
Each (model, dtype, placement) attempt runs as a separate Python process (`/tmp/t2i_worker.py`). If an attempt exhausts RAM and gets `Killed`, only that process dies, its memory is fully released, and the notebook logs it and moves on.

The fallback ladder comes from the first Kaggle runs: fp16 gave NaN/black images for Z-Image, a single T4 always ran out of memory, and CPU offload got the kernel killed. The first attempt that loads **and** gives a non-black image wins.

**Kaggle, 2× T4:**
1. bf16 split across both GPUs
2. fp16 split across both GPUs
3. 4-bit NF4 (bf16)
4. 4-bit NF4 (fp16)

**Free Colab, 1× T4, 12.7 GB RAM:**
1. 4-bit NF4 (bf16)
2. 4-bit NF4 (fp16)

In [ ]:
%%writefile /tmp/t2i_worker.py
# One attempt = one process: python t2i_worker.py <smoke|reel> <model> <dtype> <placement> <out_dir>
import sys, json, time, numpy as np, torch, diffusers as D
mode, name, dtype_s, placement, out = sys.argv[1:6]
C = json.load(open(f"{out}/config.json")); cfg = C["MODELS"][name]; dtype = getattr(torch, dtype_s)
n_gpu = torch.cuda.device_count()

def load():
    cls = getattr(D, cfg["cls"], None) or D.DiffusionPipeline
    kw = dict(torch_dtype=dtype, low_cpu_mem_usage=True)
    if placement == "balanced": kw["device_map"] = "balanced"
    if placement == "nf4":
        from diffusers.quantizers import PipelineQuantizationConfig
        kw["quantization_config"] = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs=dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype),
            components_to_quantize=cfg["quant"])
    pipe = cls.from_pretrained(cfg["repo"], **kw)
    if placement in ("gpu", "nf4"): pipe.to("cuda")
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"): pipe.vae.enable_tiling()
    pipe.set_progress_bar_config(disable=True)
    return pipe

def gen(pipe, prompt, seed):
    t = time.time()
    img = pipe(prompt=prompt, width=C["W"], height=C["H"], num_inference_steps=cfg["steps"],
               guidance_scale=cfg["guidance"], generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    return img, round(time.time() - t, 1)

def bad(img):
    return float(np.asarray(img).astype(np.float32).std()) < 2.0   # black / flat / NaN-cast output

t = time.time(); pipe = load(); load_s = round(time.time() - t, 1)
if mode == "smoke":
    gen(pipe, C["STYLE"] + C["SMOKE_PROMPT"], C["SEED"])          # warm-up
    img, secs = gen(pipe, C["STYLE"] + C["SMOKE_PROMPT"], C["SEED"])
    res = dict(load_s=load_s, seconds_per_image=secs, black_or_nan=bad(img),
               peak_vram_gb=round(max(torch.cuda.max_memory_allocated(i) for i in range(n_gpu)) / 2**30, 2))
    if not res["black_or_nan"]:
        img.save(f"{out}/smoke/{name}.png"); res["image"] = f"{name}.png"
else:
    res = []
    for i, (sid, shot) in enumerate(C["SHOTS"]):
        img, secs = gen(pipe, C["STYLE"] + shot, C["SEED"] + i)
        img.save(f"{out}/reel/{sid}_raw.png")
        res.append(dict(id=sid, prompt=C["STYLE"] + shot, seed=C["SEED"] + i, seconds=secs, black_or_nan=bad(img)))
        print(sid, secs, flush=True)
print("RESULT " + json.dumps(res), flush=True)

In [ ]:
import subprocess, sys, shutil
json.dump(dict(MODELS=MODELS, W=W, H=H, SEED=SEED, STYLE=STYLE, SMOKE_PROMPT=SMOKE_PROMPT, SHOTS=SHOTS, OUT=OUT),
          open(f"{OUT}/config.json", "w"))

def run_worker(mode, name, dtype, placement, timeout=3600):
    # Returns (result or None, error or None). A 'Killed' child only loses that one attempt.
    try:
        p = subprocess.run([sys.executable, "/tmp/t2i_worker.py", mode, name, dtype, placement, OUT],
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, f"timeout after {timeout}s"
    lines = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
    if p.returncode == 0 and lines:
        return json.loads(lines[-1][7:]), None
    if p.returncode in (-9, 137):
        return None, "Killed (out of system RAM)"
    tail = [l for l in p.stderr.strip().splitlines() if l.strip()][-6:]
    return None, f"exit {p.returncode}: " + " | ".join(tail)[-600:]

def ladder():
    if LOW_RAM or N_GPU < 2:
        return [("bfloat16", "nf4"), ("float16", "nf4")]
    return [("bfloat16", "balanced"), ("float16", "balanced"), ("bfloat16", "nf4"), ("float16", "nf4")]
print("ladder:", ladder())

In [ ]:
# 2. Smoke test: one image per model, first working ladder rung wins.
#    Saves after every model; if anything dies, re-run this cell and finished models are skipped.
SMOKE_JSON = f"{OUT}/smoke/smoke.json"
smoke = json.load(open(SMOKE_JSON))["smoke"] if os.path.exists(SMOKE_JSON) else []
done = {r["model"] for r in smoke}
for name, cfg in MODELS.items():
    if name in done or name in SKIP:
        print(f"SKIP {name} ({'done' if name in done else 'too big for this machine'})"); continue
    rec = dict(model=name, repo=cfg["repo"], licence=cfg["licence"], attempts=[])
    for dtype, placement in ladder():
        res, err = run_worker("smoke", name, dtype, placement)
        att = dict(dtype=dtype, placement=placement, **(res or {}), **({"error": err} if err else {}))
        rec["attempts"].append(att)
        if err:
            print(f"FAIL {name:15s} {dtype:8s} {placement:8s} {err[:300]}"); continue
        if res["black_or_nan"]:
            print(f"BAD  {name:15s} {dtype:8s} {placement:8s} black/NaN image, trying next rung"); continue
        att["pass"] = res["seconds_per_image"] <= PASS_SECONDS and res["peak_vram_gb"] <= PASS_VRAM_GB
        rec["result"] = att
        print(f"OK   {name:15s} {dtype:8s} {placement:8s} {res['seconds_per_image']:6.1f}s/img  "
              f"load {res['load_s']}s  peak {res['peak_vram_gb']} GB  pass={att['pass']}")
        break
    smoke.append(rec)
    json.dump(dict(env=ENV, smoke=smoke), open(SMOKE_JSON, "w"), indent=1)
    # Free disk: the three models are ~70 GB together; the winner is re-downloaded in step 3
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" + cfg["repo"].replace("/", "--")), ignore_errors=True)

In [ ]:
# Show the smoke images side by side
import matplotlib.pyplot as plt
ok = [r for r in smoke if "result" in r]
if ok:
    fig, axes = plt.subplots(1, len(ok), figsize=(5 * len(ok), 9))
    for ax, r in zip(np.atleast_1d(axes), ok):
        ax.imshow(Image.open(f"{OUT}/smoke/{r['result']['image']}")); ax.axis("off")
        ax.set_title(f"{r['model']}\n{r['result']['seconds_per_image']}s · {r['result']['peak_vram_gb']} GB · {r['result']['dtype']}/{r['result']['placement']}")
    plt.show()

## 3. Reel stills with the winner
By default the winner is the fastest model that passed. **Look at the smoke images first.** If a slower model clearly matches the house style better, set `WINNER` by hand.

In [ ]:
passed = [r for r in smoke if r.get("result", {}).get("pass")]
WINNER = None   # e.g. "flux2-klein-4b" to override
if WINNER is None:
    pool = passed or [r for r in smoke if "result" in r]
    assert pool, "No model produced a usable image: see the FAIL/BAD lines above and smoke.json"
    WINNER = min(pool, key=lambda r: r["result"]["seconds_per_image"])["model"]
win = next(r for r in smoke if r["model"] == WINNER)["result"]
cfg = MODELS[WINNER]
print("Winner:", WINNER, win["dtype"], win["placement"])
reel, err = run_worker("reel", WINNER, win["dtype"], win["placement"], timeout=7200)
assert reel, f"Reel generation failed: {err}"
for r in reel: print(f"{r['id']} {r['seconds']:5.1f}s  black/NaN={r['black_or_nan']}")

In [ ]:
# 4. Upscale x2 with Real-ESRGAN (BSD-3) via spandrel, then fit to exactly 1080x1920. Falls back to Lanczos.
import urllib.request
def upscale_all():
    try:
        from spandrel import ModelLoader
        w = "/tmp/RealESRGAN_x2plus.pth"
        if not os.path.exists(w):
            urllib.request.urlretrieve("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth", w)
        model = ModelLoader().load_from_file(w).cuda().eval().half()
        def up(img):
            x = torch.from_numpy(np.asarray(img.convert("RGB"))).permute(2, 0, 1)[None].half().cuda() / 255
            with torch.no_grad(): y = model(x).clamp(0, 1)
            return Image.fromarray((y[0].permute(1, 2, 0).float().cpu().numpy() * 255).round().astype(np.uint8))
        method = "Real-ESRGAN x2plus (BSD-3)"
    except Exception as e:
        print("Real-ESRGAN unavailable, using Lanczos:", e); up = lambda img: img; method = "Lanczos"
    for r in reel:
        img = up(Image.open(f"{OUT}/reel/{r['id']}_raw.png"))
        img.resize((1080, 1920), Image.LANCZOS).save(f"{OUT}/reel/{r['id']}.png")
        os.remove(f"{OUT}/reel/{r['id']}_raw.png")
    return method
UPSCALE = upscale_all()

secs = [r["seconds"] for r in reel]
manifest = dict(env=ENV, winner=WINNER, repo=cfg["repo"], licence=cfg["licence"], dtype=win["dtype"],
                placement=win["placement"], steps=cfg["steps"], guidance=cfg["guidance"], native_size=[W, H],
                final_size=[1080, 1920], upscale=UPSCALE, avg_seconds_per_image=round(sum(secs) / len(secs), 1),
                shots=reel, smoke=smoke)
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=1)

zpath = f"{OUT}/../t2i-test-results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for root, _, files in os.walk(OUT):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print("Avg s/img:", manifest["avg_seconds_per_image"], "| upscale:", UPSCALE, "| zip:", os.path.abspath(zpath))
if not ON_KAGGLE:
    try:
        from google.colab import files; files.download(zpath)          # works in the browser
    except Exception:
        print("Browser download unavailable (e.g. VS Code): use 'Colab: Download' on /content/t2i-test-results.zip")

In [ ]:
# Preview the 8 stills
fig, axes = plt.subplots(1, len(reel), figsize=(3 * len(reel), 6))
for ax, r in zip(axes, reel):
    ax.imshow(Image.open(f"{OUT}/reel/{r['id']}.png")); ax.axis("off"); ax.set_title(f"{r['id']} · {r['seconds']}s")
plt.show()